In [1]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import accuracy_score, f1_score


In [2]:
mlflow.set_experiment("Hyperparameter_Tuning")

2026/03/30 12:39:14 INFO mlflow.tracking.fluent: Experiment with name 'Hyperparameter_Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///C:/Users/hi/Desktop/projects/python_projects/tutorial/mlflow_playground/mlflow/mlruns/2', creation_time=1774854554161, experiment_id='2', last_update_time=1774854554161, lifecycle_stage='active', name='Hyperparameter_Tuning', tags={}, workspace='default'>

In [3]:
# Load data
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y)

In [4]:
# Took 30 sec

best_acc = 0
best_run_id = 0
param_grid = {
    "n_estimators": [10, 50,],
    "max_depth": [3, 5,],
    "min_samples_split": [2, 5],
    "criterion": ["gini",]
}

for params in ParameterGrid(param_grid):
    with mlflow.start_run():

        model = RandomForestClassifier(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average="macro")

        mlflow.log_params(params)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)
        mlflow.sklearn.log_model(model, name="model", serialization_format="skops")

        # Track best model
        if acc > best_acc:
            best_acc = acc
            best_run_id = mlflow.active_run().info.run_id


print(f"Best Accuracy: {best_acc}")
print(f"Best Run ID: {best_run_id}")

2026/03/30 12:40:14 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\hi\AppData\Local\Temp\tmpa2i8bo5i\model\model.skops, flavor: sklearn). Fall back to return ['scikit-learn==1.8.0', 'skops==0.13.0']. Set logging level to DEBUG to see the full traceback. 
2026/03/30 12:40:22 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\hi\AppData\Local\Temp\tmp54pmkkzv\model\model.skops, flavor: sklearn). Fall back to return ['scikit-learn==1.8.0', 'skops==0.13.0']. Set logging level to DEBUG to see the full traceback. 
2026/03/30 12:40:27 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\hi\AppData\Local\Temp\tmpz0gz7afo\model\model.skops, flavor: sklearn). Fall back to return ['scikit-learn==1.8.0', 'skops==0.13.0']. Set logging level to DEBUG to see the full traceback. 
2026/03/30 12:40

Best Accuracy: 0.9473684210526315
Best Run ID: e918c691407b4a51956a5ce7b7c4e457


In [5]:
model_uri = f"runs:/{best_run_id}/model"

# Register best model
registered_model = mlflow.register_model(model_uri, name="IrisClassifier")

print(f"Registered Model Version: {registered_model.version}")

Successfully registered model 'IrisClassifier'.
2026/03/30 12:41:22 WARNING mlflow.tracking._model_registry.fluent: Run with id e918c691407b4a51956a5ce7b7c4e457 has no artifacts at artifact path 'model', registering model based on models:/m-cb478737462a4b40a10c3eed365ae6b5 instead


Registered Model Version: 1


Created version '1' of model 'IrisClassifier'.


In [6]:
from mlflow.tracking import MlflowClient
client = MlflowClient()

# Move to Production
client.set_registered_model_alias(
    name="IrisClassifier",
    alias="production",
    version=registered_model.version
)


## Now run the server that serves the model:
```
python -m mlflow models serve -m models:/IrisClassifier@production --no-conda -p 5002

# Now send request to API endpoint

In [7]:
import requests
import json

url = "http://127.0.0.1:5002/invocations"
data = {
    "inputs": [
        [5.1, 3.5, 1.4, 0.2]
    ]
}

response = requests.post(url, json=data)
print(response.json())


{'predictions': [0]}
